# Reconstruction timing — one buoy, all 26 goals

Not a validation notebook — just a wall-clock number. The real reconstruction path uses *every* active goal (`reconstruct()` loops goal-by-goal, unparallelized on the GPU side) — restricting to a buoy's one closest goal, as an earlier version of this notebook did, is an approximation, not the production number. This measures one representative year across all 26 goals, bulk vs. spectral, and extrapolates linearly to the full WHACS record (fair, since cost scales with timesteps processed, not which timesteps they are).

In [1]:
# The GP prediction logging below is extremely verbose (one INFO line per
# PCA component per variable) — quiet it down so the timing output stays
# readable.
import logging
import time

import numpy as np
from utils.config import open_offshore_waves
from utils import metamodel as mm

logging.getLogger("ExactGPInterpolation").setLevel(logging.WARNING)
logging.getLogger("PCA").setLevel(logging.WARNING)

## Configuration

In [ ]:
offshore_name = "WHACS"
mda_n = 450  # must match a fitted mda_candidates entry
pca_variance = 0.99
buoy_instrument = "Sydney_Buoy"  # instrument_id from the observations GeoJSON — output site only, all 26 goals still run
sample_period = "2015"  # one representative year — measured directly; the full record below is extrapolated from it

site = (
    buoy_instrument
    if buoy_instrument.startswith("buoy_")
    else f"buoy_{buoy_instrument}"
)

whacs = open_offshore_waves(offshore_name)
t_start, t_end = whacs.time.min().values, whacs.time.max().values
full_period = slice(str(t_start)[:10], str(t_end)[:10])
n_years_full = (t_end - t_start) / np.timedelta64(365, "D")
n_years_sample = 1.0  # sample_period is one calendar year

print(f"Site          : {site}  (output only — all 26 goals are still reconstructed)")
print(
    f"Sample period : {sample_period}  (~{n_years_sample:.0f} year, measured directly)"
)
print(
    f"Full record   : {full_period.start} -> {full_period.stop}  (~{n_years_full:.1f} years, {whacs.sizes['time']:,} offshore timesteps — extrapolated from the sample)"
)

Site          : buoy_Sydney_Buoy  (output only — all 26 goals are still reconstructed)
Sample period : 2015  (~1 year, measured directly)
Full record   : 1979-01-01 -> 2023-12-31  (~45.0 years, 394,464 offshore timesteps — extrapolated from the sample)


## Check fitted models

In [ ]:
mm.require_models_all(
    mda_n, None, pca_variance=pca_variance
)  # None = every active goal

## Reconstruct — bulk vs. spectral, all 26 goals

`use_cache=False` — a genuine from-scratch number, not a cache hit from an earlier run. `goals=None` (every active goal) is the real production path; `sites=[site]` only restricts which site's *output* we keep, not which goals get reconstructed. Both modes now benefit from `RECONSTRUCT_GPU_BATCH_MONTHS_CAP=4` (re-tuned for this — see `metamodel.py`'s comment on that constant for the full story: bulk tolerates much bigger batches, spectral is the binding constraint).

In [ ]:
timings: dict[str, float] = {}
sizes: dict[str, int] = {}

for spectral, label in [(False, "Bulk metamodel"), (True, "Spectral metamodel")]:
    t0 = time.perf_counter()
    result = mm.reconstruct(
        sample_period,
        mda_n,
        spectral=spectral,
        offshore_name=offshore_name,
        goals=None,
        sites=[site],
        pca_variance=pca_variance,
        use_cache=False,
    )
    timings[label] = time.perf_counter() - t0
    sizes[label] = int(result.sizes["time"])
    print(
        f"{label:20s}: {timings[label]:8.1f}s for {sizes[label]:,} timesteps ({sample_period})"
    )

Bulk metamodel      :    349.4s for 8,760 timesteps (2015)


Spectral metamodel  :    906.7s for 8,760 timesteps (2015)


## Summary — measured (1 year, 26 goals) and extrapolated (full record)

In [ ]:
print(f"Buoy          : {site}  (all 26 goals reconstructed, one site kept)")
print(f"Sample        : {sample_period}  (~{n_years_sample:.0f} year)")
print(
    f"Full record   : {full_period.start} -> {full_period.stop}  (~{n_years_full:.1f} years)"
)
print()
scale = n_years_full / n_years_sample
for label, secs in timings.items():
    print(
        f"{label:20s}  sample: {secs / 60:6.1f} min   extrapolated (~{n_years_full:.0f}y): {secs * scale / 60:7.1f} min ({1000 * secs / sizes[label]:5.2f} ms/timestep)"
    )
print()
sample_total_min = sum(timings.values()) / 60
full_total_min = sum(timings.values()) * scale / 60
print(
    f"Total (bulk + spectral) — sample: {sample_total_min:.1f} min   extrapolated full record: {full_total_min:.0f} min (~{full_total_min / 60:.1f} h)"
)

Buoy          : buoy_Sydney_Buoy  (all 26 goals reconstructed, one site kept)
Sample        : 2015  (~1 year)
Full record   : 1979-01-01 -> 2023-12-31  (~45.0 years)

Bulk metamodel        sample:    5.8 min   extrapolated (~45y):   262.2 min (39.89 ms/timestep)
Spectral metamodel    sample:   15.1 min   extrapolated (~45y):   680.5 min (103.50 ms/timestep)

Total (bulk + spectral) — sample: 20.9 min   extrapolated full record: 943 min (~15.7 h)
